<a href="https://colab.research.google.com/github/sahar-mariam/kannada-sentiment-analysis/blob/main/machine_trans_phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install Dependencies
!git clone https://github.com/VarunGumma/IndicTransToolkit.git
%cd IndicTransToolkit
!pip install --editable ./
!pip install transformers datasets sentencepiece evaluate sacremoses --quiet

# 2. Imports
import json
import pandas as pd
import torch
from tqdm import tqdm
from datasets import Dataset
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 3. Load Model and Tokenizer
model_name = "ai4bharat/indictrans2-indic-en-1B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(DEVICE)
ip = IndicProcessor(inference=True)

Cloning into 'IndicTransToolkit'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 222 (delta 61), reused 97 (delta 47), pack-reused 95 (from 1)
Receiving objects: 100% (222/222), 4.38 MiB | 20.68 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/IndicTransToolkit
Obtaining file:///content/IndicTransToolkit
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Cloning https://github.com/VarunGumma/indic_nlp_library.git to /tmp/pip-install-rqkbr325/indic-nlp-library-it2_dfa1d731f4814d9ba019fce9a30ae0a1
  Running command git clone --filter=blob:none --quiet https://github.com/VarunGumma/indic_nlp_library.git /tmp/pip-install-rqkbr325/indic-nlp-library-it2_dfa1d731f4814d9ba019fce9a30ae0a1
  Resolved https://github.com/Varun

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenization_indictrans.py:   0%|          | 0.00/8.13k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- configuration_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_indictrans.py:   0%|          | 0.00/79.8k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

In [ ]:
# 4. Load Dataset (.jsonl format)
def load_and_validate_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        lines = [json.loads(line) for line in f if line.strip()]
    return Dataset.from_list(lines)

raw_data = load_and_validate_dataset("/content/ksa.jsonl")

# 5. Translation Function (Kannada → English)
def translate_kannada_to_english(texts, batch_size=8):
    translations = []
    src_lang, tgt_lang = "kan_Knda", "eng_Latn"
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating Kannada → English"):
        batch_texts = texts[i:i+batch_size]
        batch = ip.preprocess_batch(batch_texts, src_lang=src_lang, tgt_lang=tgt_lang)
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        with torch.no_grad():
           outputs = model.generate(
               **inputs,
               use_cache=True,
               min_length=0,
               max_length=256,
               num_beams=5,
               repetition_penalty=1.2,
               length_penalty=1.0,
               early_stopping=True
               )
        with tokenizer.as_target_tokenizer():
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations += ip.postprocess_batch(decoded, lang=tgt_lang)
    return translations

In [ ]:
# 6. Add Translated Column
raw_data = raw_data.add_column("en_translated", translate_kannada_to_english(raw_data["kn"]))

# 7. Load Sentiment Pipeline
sentiment_pipe = pipeline("text-classification", model="distilbert-base-uncased-finetuned-sst-2-english", device=0 if torch.cuda.is_available() else -1)
predictions = sentiment_pipe(raw_data["en_translated"], batch_size=32)
predicted = [p["label"].lower() for p in predictions]
raw_data = raw_data.add_column("predicted_sentiment", predicted)

# 8. Evaluation
print("\nEvaluation Report:")
print(classification_report(raw_data["label"], raw_data["predicted_sentiment"], digits=3))
accuracy = accuracy_score(raw_data["label"], raw_data["predicted_sentiment"])
f1 = f1_score(raw_data["label"], raw_data["predicted_sentiment"], average="weighted")
print(f"\nAccuracy: {accuracy:.4f} | Weighted F1: {f1:.4f}")


Translating Kannada → English: 100%|██████████| 106/106 [01:29<00:00,  1.18it/s]
Device set to use cuda:0



Evaluation Report:
              precision    recall  f1-score   support

    negative      0.632     0.950     0.759       262
     neutral      0.000     0.000     0.000       304
    positive      0.578     0.938     0.715       276

    accuracy                          0.603       842
   macro avg      0.403     0.630     0.492       842
weighted avg      0.386     0.603     0.471       842


Accuracy: 0.6033 | Weighted F1: 0.4707


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# 9. Inference Method
def predict_and_translate(input_text):
    preprocessed = ip.preprocess_batch([input_text], src_lang="kan_Knda", tgt_lang="eng_Latn")
    inputs = tokenizer(preprocessed, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        output = model.generate(**inputs, num_beams=5, max_length=256)
    with tokenizer.as_target_tokenizer():
        decoded = tokenizer.batch_decode(output, skip_special_tokens=True)
    translated = ip.postprocess_batch(decoded, lang="eng_Latn")[0]

    sentiment_result = sentiment_pipe(translated)[0]
    sentiment = sentiment_result["label"].lower()
    confidence = round(sentiment_result["score"], 4)

    return translated, sentiment, confidence

# 10. User Input for Real-Time Inference
user_text = input("Enter Kannada text: ")
translated, sentiment, confidence = predict_and_translate(user_text)
print(f"\nTranslated Text: {translated}\nPredicted Sentiment: {sentiment}\nConfidence Score: {confidence}")

# 11. Save Final Results
pd.DataFrame({
    "en": raw_data["en"],
    "kn": raw_data["kn"],
    "label": raw_data["label"],
    "predicted_sentiment": raw_data["predicted_sentiment"],
    "en_translated": raw_data["en_translated"],
}).to_csv("/content/translation_sentiment_results.csv", index=False)

print("\n✅ Complete pipeline (IndicTrans2 + sentiment) finished and results saved.")


Enter Kannada text: ತಾಯಿಗೆ ತನ್ನ ಮಗನ ಬಗ್ಗೆ ನಿರಾಶೆಯಾಯಿತು.

Translated Text: The mother was disappointed in her son.
Predicted Sentiment: negative
Confidence Score: 0.9992

✅ Complete pipeline (IndicTrans2 + sentiment) finished and results saved.


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
